# Part 2: How Real VLAs Represent Actions

## Notebook 6 — SmolVLA: Cross-Attention Action Expert

SmolVLA (HuggingFace, 2025) is a lightweight VLA built on SmolVLM2 (500M). Unlike pi0 which passes action tokens directly into the transformer, SmolVLA uses **cross-attention** between the VLM latents and the action expert.


### 1. Load SmolVLA configuration

SmolVLA balances size and performance. The action expert is configurable in width and uses cross-attention to attend to VLM outputs.


In [ ]:
from lerobot.policies.smolvla.configuration_smolvla import SmolVLAConfig

cfg = SmolVLAConfig()
print(f"Policy type: SmolVLA (Lightweight VLA)")
print(f"Chunk size:          {cfg.chunk_size}")  # 50
print(f"Action steps:        {cfg.n_action_steps}")  # 50
print(f"Max action dim:      {cfg.max_action_dim}")  # 32
print(f"Expert width mult:   {cfg.expert_width_multiplier}")  # 0.75
print(f"Attention mode:      {cfg.attention_mode}")  # cross_attn
print(f"Train expert only:   {cfg.train_expert_only}")  # True
print(f"Num expert layers:   {cfg.num_expert_layers}")  # -1 (all)


### 2. Cross-attention vs direct concatenation

pi0: action tokens go into the main transformer alongside image/text tokens.
SmolVLA: action expert is a SEPARATE transformer that cross-attends to the VLM's output latents.

This decoupling means the VLM doesn't need to be modified for action generation. The action expert is a bolt-on module.


In [ ]:
# Architecture comparison
print("pi0 Architecture:")
print("  [Image Tokens] [Text Tokens] [State] [Action Tokens]")
print("       ↓              ↓          ↓         ↓")
print("  ┌──────────────── VLM + Expert ────────────────┐")
print("  │  All tokens processed together               │")
print("  │  Action expert is a subset of layers         │")
print("  └──────────────────────────────────────────────┘")

print("\nSmolVLA Architecture:")
print("  [Image Tokens] [Text Tokens] [State]")
print("       ↓              ↓          ↓")
print("  ┌──────────────── VLM ───────────┐")
print("  │  Processed into latents        │──→ VLM output
print("  └────────────────────────────────┘")
print("                    ↓")
print("  ┌── Action Expert (cross_attn) ──┐")
print("  │  Cross-attends to VLM latents  │")
print("  │  action_in_proj / out_proj     │──→ Actions
print("  └────────────────────────────────┘")


### 3. Action representation: continuous regression

Like all models in Part 2, SmolVLA outputs continuous actions. The action_in_proj/action_out_proj MLPs project to/from the expert's hidden dimension.


In [ ]:
# From modeling_smolvla.py:
# self.action_in_proj = nn.Linear(max_action_dim, expert_hidden_size)
# self.action_out_proj = nn.Linear(expert_hidden_size, max_action_dim)

# Forward pass:
# 1. VLM processes images + text → latent representations
# 2. Action expert cross-attends to VLM latents
# 3. action_in_proj projects noisy actions → expert
# 4. Expert transformer processes → action predictions
# 5. action_out_proj → continuous action chunk (50 × 7)

print("SmolVLA Action Path:")
print("  noise → action_in_proj → [B, 50, expert_hidden]")
print("  VLM latents → cross_attn(query=action, key/value=VLM_output)")
print("  expert hidden → action_out_proj → [B, 50, 7]")


### 4. Why SmolVLA is efficient

- SmolVLM2-500M backbone (vs pi0's PaliGemma 2B)
- `expert_width_multiplier=0.75` reduces expert width
- `train_expert_only=True` by default — only ~300M trainable
- Runs on consumer GPUs at 30 Hz (RTX 3090)


In [ ]:
# Efficiency comparison
print("Model size comparison (for inference):")
print("  pi0:         ~3B params (PaliGemma 2B + Gemma 300M expert)")
print("  SmolVLA:     ~500M params (SmolVLM2 + shrunk expert)")
print("  pi0-FAST:    ~3B params (PaliGemma 2B, no separate expert)")

print("Inference speed (approx):")
print("  SmolVLA:     30 Hz (RTX 3090)")
print("  pi0:         10-25 Hz (RTX 4090 / A100)")
print("  pi0-FAST:    ~5 Hz (autoregressive decoding)")


### Key Takeaway

SmolVLA uses cross-attention to decouple the VLM from the action expert. Actions are continuous, the expert is a separate transformer, and efficiency comes from the small backbone (500M) and reduced expert width. Still no tokenization.
